In [1]:
import pandas as pd
import numpy as np
import ast
import re
import os

from pathlib import Path
from itertools import combinations

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

In [2]:
events_df = pd.read_csv("../data/processed/extracted_events_v2.csv")

print("Total extracted events:", len(events_df))
print("Columns:", events_df.columns.tolist())

display(events_df.head())

Total extracted events: 42
Columns: ['article_id', 'sentence_id', 'publication_date', 'original_sentence', 'event_trigger', 'action', 'subject', 'dependency', 'head_action', 'predicate_type', 'pos_tag', 'subject_source', 'object', 'agent', 'passive', 'negation', 'temporal_entities', 'location_entities']


,article_id,sentence_id,publication_date,original_sentence,event_trigger,action,subject,dependency,head_action,predicate_type,pos_tag,subject_source,object,agent,passive,negation,temporal_entities,location_entities
0,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...",began,begin,['Continuous rainfall'],ROOT,begin,main,VBD,direct,[],[],False,False,"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
1,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...",causing,cause,['Continuous rainfall'],advcl,begin,subordinate,VBG,inherited_subordinate,[],[],False,False,"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
2,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...",flood,flood,['several streets'],ccomp,cause,subordinate,VB,direct,[],[],False,False,"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
3,article_01,2,2026-08-14,The city weather office recorded 118 millimetres of rain in twelve hours.,recorded,record,['The city weather office'],ROOT,record,main,VBD,direct,['118 millimetres'],[],False,False,['twelve hours'],[]
4,article_01,3,2026-08-14,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,received,receive,['Emergency services'],ROOT,receive,main,VBD,direct,['their first major flooding calls'],[],False,False,"['approximately 5:30 a.m.', 'August 14']",[]


In [3]:
print("Data types:\n")
display(events_df.dtypes)

print("\nMissing values:\n")
display(events_df.isnull().sum())

print("\nSample events:\n")
display(
    events_df[
        [
            'article_id',
            'original_sentence',
            'event_trigger',
            'action',
            'subject',
            'object',
            'temporal_entities',
            'location_entities'
        ]
    ].head(10)
)

Data types:



article_id             str
sentence_id          int64
publication_date       str
original_sentence      str
event_trigger          str
action                 str
subject                str
dependency             str
head_action            str
predicate_type         str
pos_tag                str
subject_source         str
object                 str
agent                  str
passive               bool
negation              bool
temporal_entities      str
location_entities      str
dtype: object


Missing values:



article_id           0
sentence_id          0
publication_date     0
original_sentence    0
event_trigger        0
action               0
subject              0
dependency           0
head_action          0
predicate_type       0
pos_tag              0
subject_source       0
object               0
agent                0
passive              0
negation             0
temporal_entities    0
location_entities    0
dtype: int64


Sample events:



,article_id,original_sentence,event_trigger,action,subject,object,temporal_entities,location_entities
0,article_01,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...",began,begin,['Continuous rainfall'],[],"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
1,article_01,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...",causing,cause,['Continuous rainfall'],[],"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
2,article_01,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...",flood,flood,['several streets'],[],"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
3,article_01,The city weather office recorded 118 millimetres of rain in twelve hours.,recorded,record,['The city weather office'],['118 millimetres'],['twelve hours'],[]
4,article_01,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,received,receive,['Emergency services'],['their first major flooding calls'],"['approximately 5:30 a.m.', 'August 14']",[]
5,article_01,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",entered,enter,['Water'],['ground - floor homes'],[],"['the Old Bridge', 'the Riverside neighbourhood']"
6,article_01,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",diverted,divert,['traffic'],[],[],"['the Old Bridge', 'the Riverside neighbourhood']"
7,article_01,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,opened,open,['The municipal corporation'],['two temporary shelters'],[],['Central High School and the Riverside Community Hall']
8,article_01,Officials said that 46 residents had been moved to the shelters by 10 a.m.,moved,move,['46 residents'],[],['10 a.m.'],[]
9,article_01,The weather office warned that further rainfall was expected through Saturday.,warned,warn,['The weather office'],[],['through Saturday'],[]


In [4]:
list_columns = [
    'subject',
    'object',
    'agent',
    'temporal_entities',
    'location_entities'
]

def parse_list(value):
    if pd.isna(value) or value == '':
        return []
    
    if isinstance(value, list):
        return value
    
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else [str(parsed)]
    except (ValueError, SyntaxError):
        return [value]

for col in list_columns:
    events_df[col] = events_df[col].apply(parse_list)

display(events_df[list_columns].head(10))

,subject,object,agent,temporal_entities,location_entities
0,[Continuous rainfall],[],[],"[August 13, early Friday morning]","[Riverton, the eastern part of the city]"
1,[Continuous rainfall],[],[],"[August 13, early Friday morning]","[Riverton, the eastern part of the city]"
2,[several streets],[],[],"[August 13, early Friday morning]","[Riverton, the eastern part of the city]"
3,[The city weather office],[118 millimetres],[],[twelve hours],[]
4,[Emergency services],[their first major flooding calls],[],"[approximately 5:30 a.m., August 14]",[]
5,[Water],[ground - floor homes],[],[],"[the Old Bridge, the Riverside neighbourhood]"
6,[traffic],[],[],[],"[the Old Bridge, the Riverside neighbourhood]"
7,[The municipal corporation],[two temporary shelters],[],[],[Central High School and the Riverside Community Hall]
8,[46 residents],[],[],[10 a.m.],[]
9,[The weather office],[],[],[through Saturday],[]


In [5]:
def normalize_text(text):
    if not isinstance(text, str):
        return ''
    
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text


def normalize_list(items):
    return [normalize_text(item) for item in items if normalize_text(item)]


events_df['action_norm'] = events_df['action'].apply(normalize_text)

events_df['subject_norm'] = events_df['subject'].apply(normalize_list)

events_df['object_norm'] = events_df['object'].apply(normalize_list)

events_df['location_norm'] = events_df['location_entities'].apply(normalize_list)

events_df['temporal_norm'] = events_df['temporal_entities'].apply(normalize_list)

display(
    events_df[
        [
            'article_id',
            'action_norm',
            'subject_norm',
            'object_norm',
            'location_norm',
            'temporal_norm'
        ]
    ].head(10)
)

,article_id,action_norm,subject_norm,object_norm,location_norm,temporal_norm
0,article_01,begin,[continuous rainfall],[],"[riverton, the eastern part of the city]","[august 13, early friday morning]"
1,article_01,cause,[continuous rainfall],[],"[riverton, the eastern part of the city]","[august 13, early friday morning]"
2,article_01,flood,[several streets],[],"[riverton, the eastern part of the city]","[august 13, early friday morning]"
3,article_01,record,[the city weather office],[118 millimetres],[],[twelve hours]
4,article_01,receive,[emergency services],[their first major flooding calls],[],"[approximately 530 am, august 14]"
5,article_01,enter,[water],[ground floor homes],"[the old bridge, the riverside neighbourhood]",[]
6,article_01,divert,[traffic],[],"[the old bridge, the riverside neighbourhood]",[]
7,article_01,open,[the municipal corporation],[two temporary shelters],[central high school and the riverside community hall],[]
8,article_01,move,[46 residents],[],[],[10 am]
9,article_01,warn,[the weather office],[],[],[through saturday]


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

event_embeddings = model.encode(
    events_df['original_sentence'].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", event_embeddings.shape)

W1115 17:44:25.727000 49282 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (42, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(event_embeddings)

event_pairs = []

for i, j in combinations(range(len(events_df)), 2):

    if events_df.iloc[i]['article_id'] == events_df.iloc[j]['article_id']:
        continue

    event_pairs.append({
        'event_1': i,
        'event_2': j,
        'article_1': events_df.iloc[i]['article_id'],
        'article_2': events_df.iloc[j]['article_id'],
        'sentence_1': events_df.iloc[i]['original_sentence'],
        'sentence_2': events_df.iloc[j]['original_sentence'],
        'semantic_similarity': similarity_matrix[i, j]
    })

similarity_df = pd.DataFrame(event_pairs)

similarity_df = similarity_df.sort_values(
    by='semantic_similarity',
    ascending=False
).reset_index(drop=True)

print("Total cross-document pairs:", len(similarity_df))

display(similarity_df.head(20))

Total cross-document pairs: 585


,event_1,event_2,article_1,article_2,sentence_1,sentence_2,semantic_similarity
0,7,20,article_01,article_02,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,0.899079
1,7,21,article_01,article_02,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,0.899079
2,3,31,article_01,article_03,The city weather office recorded 118 millimetres of rain in twelve hours.,The weather office reported that the city had received 118 millimetres of rain in the twelve hours leading up to Friday morning.,0.889799
3,3,30,article_01,article_03,The city weather office recorded 118 millimetres of rain in twelve hours.,The weather office reported that the city had received 118 millimetres of rain in the twelve hours leading up to Friday morning.,0.889799
4,20,32,article_02,article_03,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,The municipal corporation said that 60 residents had used the temporary shelters at Central High School and Riverside Community Hall.,0.841379
5,21,32,article_02,article_03,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,The municipal corporation said that 60 residents had used the temporary shelters at Central High School and Riverside Community Hall.,0.841379
6,7,32,article_01,article_03,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,The municipal corporation said that 60 residents had used the temporary shelters at Central High School and Riverside Community Hall.,0.829408
7,0,13,article_01,article_02,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...","Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.",0.771811
8,2,13,article_01,article_02,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...","Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.",0.771811
9,2,14,article_01,article_02,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...","Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.",0.771811


In [8]:
print("Semantic similarity statistics:")
display(similarity_df['semantic_similarity'].describe())

print("\nTop 20 candidate pairs with structured attributes:")

for _, row in similarity_df.head(20).iterrows():

    i = row['event_1']
    j = row['event_2']

    e1 = events_df.iloc[i]
    e2 = events_df.iloc[j]

    print("-" * 100)
    print(f"Similarity: {row['semantic_similarity']:.3f}")

    print(f"\nEvent 1: {e1['original_sentence']}")
    print(f"Action: {e1['action_norm']} | Subject: {e1['subject_norm']}")
    print(f"Object: {e1['object_norm']} | Location: {e1['location_norm']}")

    print(f"\nEvent 2: {e2['original_sentence']}")
    print(f"Action: {e2['action_norm']} | Subject: {e2['subject_norm']}")
    print(f"Object: {e2['object_norm']} | Location: {e2['location_norm']}")

Semantic similarity statistics:


count    585.000000
mean       0.243573
std        0.191072
min       -0.060006
25%        0.119237
50%        0.192355
75%        0.323000
max        0.899079
Name: semantic_similarity, dtype: float64


Top 20 candidate pairs with structured attributes:
----------------------------------------------------------------------------------------------------
Similarity: 0.899

Event 1: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
Action: open | Subject: ['the municipal corporation']
Object: ['two temporary shelters'] | Location: ['central high school and the riverside community hall']

Event 2: The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.
Action: confirm | Subject: ['the municipal corporation']
Object: [] | Location: ['central high school and riverside community hall']
----------------------------------------------------------------------------------------------------
Similarity: 0.899

Event 1: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
Action: open | Subject: ['the muni

In [9]:
def jaccard_similarity(list1, list2):
    set1 = set(" ".join(list1).split())
    set2 = set(" ".join(list2).split())

    if not set1 or not set2:
        return 0.0

    return len(set1 & set2) / len(set1 | set2)


def attribute_similarity(row, col):
    i = row['event_1']
    j = row['event_2']

    e1 = events_df.iloc[i]
    e2 = events_df.iloc[j]

    return jaccard_similarity(e1[col], e2[col])


similarity_df['subject_similarity'] = similarity_df.apply(
    lambda row: attribute_similarity(row, 'subject_norm'), axis=1
)

similarity_df['object_similarity'] = similarity_df.apply(
    lambda row: attribute_similarity(row, 'object_norm'), axis=1
)

similarity_df['location_similarity'] = similarity_df.apply(
    lambda row: attribute_similarity(row, 'location_norm'), axis=1
)

similarity_df['temporal_similarity'] = similarity_df.apply(
    lambda row: attribute_similarity(row, 'temporal_norm'), axis=1
)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'semantic_similarity',
            'subject_similarity',
            'object_similarity',
            'location_similarity',
            'temporal_similarity'
        ]
    ].head(20)
)

,event_1,event_2,semantic_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity
0,7,20,0.899079,1.000000,0.0,0.875000,0.00
1,7,21,0.899079,0.000000,0.0,0.875000,0.00
2,3,31,0.889799,0.000000,0.0,0.000000,0.50
3,3,30,0.889799,0.500000,1.0,0.000000,0.50
4,20,32,0.841379,0.000000,0.0,1.000000,0.00
5,21,32,0.841379,0.000000,0.0,1.000000,0.00
6,7,32,0.829408,0.000000,0.5,0.875000,0.00
7,0,13,0.771811,0.333333,0.0,0.166667,0.25
8,2,13,0.771811,0.000000,0.0,0.166667,0.25
9,2,14,0.771811,0.000000,0.0,0.166667,0.25


In [10]:
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('omw-1.4')


def action_similarity(action1, action2):

    if not action1 or not action2:
        return 0.0

    if action1 == action2:
        return 1.0

    synsets1 = wordnet.synsets(action1, pos=wordnet.VERB)
    synsets2 = wordnet.synsets(action2, pos=wordnet.VERB)

    if not synsets1 or not synsets2:
        return 0.0

    scores = []

    for syn1 in synsets1:
        for syn2 in synsets2:
            score = syn1.path_similarity(syn2)

            if score is not None:
                scores.append(score)

    return max(scores) if scores else 0.0


similarity_df['action_similarity'] = similarity_df.apply(
    lambda row: action_similarity(
        events_df.iloc[row['event_1']]['action_norm'],
        events_df.iloc[row['event_2']]['action_norm']
    ),
    axis=1
)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'semantic_similarity',
            'action_similarity'
        ]
    ].head(20)
)

[nltk_data] Downloading package wordnet to /Users/kavya/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/kavya/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,event_1,event_2,semantic_similarity,action_similarity
0,7,20,0.899079,0.333333
1,7,21,0.899079,0.333333
2,3,31,0.889799,0.333333
3,3,30,0.889799,0.333333
4,20,32,0.841379,0.333333
5,21,32,0.841379,0.333333
6,7,32,0.829408,0.333333
7,0,13,0.771811,0.333333
8,2,13,0.771811,0.166667
9,2,14,0.771811,0.250000


In [11]:
def create_event_representation(row):

    subject = ", ".join(row['subject_norm']) or "unknown"
    obj = ", ".join(row['object_norm']) or "unknown"

    return (
        f"Action: {row['action_norm']}. "
        f"Subject: {subject}. "
        f"Object: {obj}. "
        f"Context: {row['original_sentence']}"
    )


events_df['event_representation'] = events_df.apply(
    create_event_representation,
    axis=1
)

event_embeddings = model.encode(
    events_df['event_representation'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Event embeddings shape:", event_embeddings.shape)

display(
    events_df[
        ['event_representation']
    ].head(10)
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Event embeddings shape: (42, 384)


,event_representation
0,"Action: begin. Subject: continuous rainfall. Object: unknown. Context: Continuous rainfall began across Riverton late on August 13, causing severa..."
1,"Action: cause. Subject: continuous rainfall. Object: unknown. Context: Continuous rainfall began across Riverton late on August 13, causing severa..."
2,"Action: flood. Subject: several streets. Object: unknown. Context: Continuous rainfall began across Riverton late on August 13, causing several st..."
3,Action: record. Subject: the city weather office. Object: 118 millimetres. Context: The city weather office recorded 118 millimetres of rain in tw...
4,Action: receive. Subject: emergency services. Object: their first major flooding calls. Context: Emergency services received their first major flo...
5,"Action: enter. Subject: water. Object: ground floor homes. Context: Water entered ground-floor homes in the Riverside neighbourhood, and traffic w..."
6,"Action: divert. Subject: traffic. Object: unknown. Context: Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diver..."
7,Action: open. Subject: the municipal corporation. Object: two temporary shelters. Context: The municipal corporation opened two temporary shelters...
8,Action: move. Subject: 46 residents. Object: unknown. Context: Officials said that 46 residents had been moved to the shelters by 10 a.m.
9,Action: warn. Subject: the weather office. Object: unknown. Context: The weather office warned that further rainfall was expected through Saturday.


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(event_embeddings)

similarity_df['semantic_similarity'] = similarity_df.apply(
    lambda row: similarity_matrix[
        int(row['event_1']),
        int(row['event_2'])
    ],
    axis=1
)

similarity_df = similarity_df.sort_values(
    by='semantic_similarity',
    ascending=False
).reset_index(drop=True)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'article_1',
            'article_2',
            'semantic_similarity',
            'action_similarity',
            'subject_similarity',
            'object_similarity',
            'location_similarity',
            'temporal_similarity'
        ]
    ].head(20)
)

,event_1,event_2,article_1,article_2,semantic_similarity,action_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity
0,1,29,article_01,article_03,0.844869,1.000000,0.000000,0.0,0.166667,0.100
1,3,30,article_01,article_03,0.844065,0.333333,0.500000,1.0,0.000000,0.500
2,7,21,article_01,article_02,0.825796,0.333333,0.000000,0.0,0.875000,0.000
3,13,27,article_02,article_03,0.810595,1.000000,0.000000,0.0,1.000000,0.375
4,2,27,article_01,article_03,0.795553,0.166667,0.333333,0.0,0.166667,0.100
5,2,29,article_01,article_03,0.793422,0.200000,0.000000,0.0,0.166667,0.100
6,21,32,article_02,article_03,0.788178,0.333333,0.000000,0.0,1.000000,0.000
7,0,13,article_01,article_02,0.784602,0.333333,0.333333,0.0,0.166667,0.250
8,9,25,article_01,article_02,0.776541,0.200000,1.000000,0.0,0.000000,0.250
9,7,32,article_01,article_03,0.775200,0.333333,0.000000,0.5,0.875000,0.000


In [13]:
similarity_df['combined_similarity'] = (
    0.45 * similarity_df['semantic_similarity'] +
    0.20 * similarity_df['action_similarity'] +
    0.15 * similarity_df['subject_similarity'] +
    0.10 * similarity_df['object_similarity'] +
    0.05 * similarity_df['location_similarity'] +
    0.05 * similarity_df['temporal_similarity']
)

similarity_df = similarity_df.sort_values(
    by='combined_similarity',
    ascending=False
).reset_index(drop=True)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'semantic_similarity',
            'action_similarity',
            'subject_similarity',
            'object_similarity',
            'location_similarity',
            'temporal_similarity',
            'combined_similarity'
        ]
    ].head(25)
)

,event_1,event_2,semantic_similarity,action_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity,combined_similarity
0,3,30,0.844065,0.333333,0.500000,1.0,0.000000,0.500,0.646496
1,13,27,0.810595,1.000000,0.000000,0.0,1.000000,0.375,0.633518
2,7,20,0.773597,0.333333,1.000000,0.0,0.875000,0.000,0.608535
3,1,29,0.844869,1.000000,0.000000,0.0,0.166667,0.100,0.593524
4,16,39,0.770240,0.333333,1.000000,0.0,0.000000,0.000,0.563275
5,9,25,0.776541,0.200000,1.000000,0.0,0.000000,0.250,0.551943
6,13,26,0.698233,0.333333,0.500000,0.0,1.000000,0.375,0.524621
7,7,32,0.775200,0.333333,0.000000,0.5,0.875000,0.000,0.509257
8,23,41,0.579168,1.000000,0.250000,0.0,0.000000,0.000,0.498126
9,0,13,0.784602,0.333333,0.333333,0.0,0.166667,0.250,0.490571


In [14]:
for _, row in similarity_df.head(25).iterrows():

    i = int(row['event_1'])
    j = int(row['event_2'])

    e1 = events_df.iloc[i]
    e2 = events_df.iloc[j]

    print("=" * 110)
    print(f"Event IDs: {i} and {j}")
    print(f"Articles: {e1['article_id']} and {e2['article_id']}")
    print(f"Combined similarity: {row['combined_similarity']:.3f}")

    print(f"\nEVENT 1")
    print("Action:", e1['action_norm'])
    print("Subject:", e1['subject_norm'])
    print("Object:", e1['object_norm'])
    print("Location:", e1['location_norm'])
    print("Temporal:", e1['temporal_norm'])
    print("Sentence:", e1['original_sentence'])

    print(f"\nEVENT 2")
    print("Action:", e2['action_norm'])
    print("Subject:", e2['subject_norm'])
    print("Object:", e2['object_norm'])
    print("Location:", e2['location_norm'])
    print("Temporal:", e2['temporal_norm'])
    print("Sentence:", e2['original_sentence'])

Event IDs: 3 and 30
Articles: article_01 and article_03
Combined similarity: 0.646

EVENT 1
Action: record
Subject: ['the city weather office']
Object: ['118 millimetres']
Location: []
Temporal: ['twelve hours']
Sentence: The city weather office recorded 118 millimetres of rain in twelve hours.

EVENT 2
Action: receive
Subject: ['the city']
Object: ['118 millimetres']
Location: ['the twelve hours leading up to friday morning']
Temporal: ['friday', 'morning', 'twelve hours']
Sentence: The weather office reported that the city had received 118 millimetres of rain in the twelve hours leading up to Friday morning.
Event IDs: 13 and 27
Articles: article_02 and article_03
Combined similarity: 0.634

EVENT 1
Action: continue
Subject: ['heavy rainfall']
Object: []
Location: ['riverton']
Temporal: ['friday august 14', 'the day']
Sentence: Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.

EVENT 2
Action: continue
Subject: 

In [15]:
def create_action_context(row):

    subject = " ".join(row['subject_norm'])
    action = row['action_norm']
    obj = " ".join(row['object_norm'])
    location = " ".join(row['location_norm'])

    context = f"{subject} {action} {obj} {location}"

    return context.strip()


events_df['action_context'] = events_df.apply(
    create_action_context,
    axis=1
)

display(
    events_df[
        [
            'action_norm',
            'subject_norm',
            'object_norm',
            'location_norm',
            'action_context'
        ]
    ].head(10)
)

,action_norm,subject_norm,object_norm,location_norm,action_context
0,begin,[continuous rainfall],[],"[riverton, the eastern part of the city]",continuous rainfall begin riverton the eastern part of the city
1,cause,[continuous rainfall],[],"[riverton, the eastern part of the city]",continuous rainfall cause riverton the eastern part of the city
2,flood,[several streets],[],"[riverton, the eastern part of the city]",several streets flood riverton the eastern part of the city
3,record,[the city weather office],[118 millimetres],[],the city weather office record 118 millimetres
4,receive,[emergency services],[their first major flooding calls],[],emergency services receive their first major flooding calls
5,enter,[water],[ground floor homes],"[the old bridge, the riverside neighbourhood]",water enter ground floor homes the old bridge the riverside neighbourhood
6,divert,[traffic],[],"[the old bridge, the riverside neighbourhood]",traffic divert the old bridge the riverside neighbourhood
7,open,[the municipal corporation],[two temporary shelters],[central high school and the riverside community hall],the municipal corporation open two temporary shelters central high school and the riverside community hall
8,move,[46 residents],[],[],46 residents move
9,warn,[the weather office],[],[],the weather office warn


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

action_context_embeddings = model.encode(
    events_df['action_context'].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

action_context_matrix = cosine_similarity(action_context_embeddings)

similarity_df['contextual_action_similarity'] = similarity_df.apply(
    lambda row: action_context_matrix[
        int(row['event_1']),
        int(row['event_2'])
    ],
    axis=1
)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'action_similarity',
            'contextual_action_similarity',
            'semantic_similarity'
        ]
    ].head(25)
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

,event_1,event_2,action_similarity,contextual_action_similarity,semantic_similarity
0,3,30,0.333333,0.650604,0.844065
1,13,27,1.000000,0.637284,0.810595
2,7,20,0.333333,0.775490,0.773597
3,1,29,1.000000,0.610771,0.844869
4,16,39,0.333333,0.832950,0.770240
5,9,25,0.200000,0.684790,0.776541
6,13,26,0.333333,0.830932,0.698233
7,7,32,0.333333,0.834307,0.775200
8,23,41,1.000000,0.374972,0.579168
9,0,13,0.333333,0.820786,0.784602


In [17]:
similarity_df['contextual_score'] = (
    0.5 * similarity_df['semantic_similarity'] +
    0.5 * similarity_df['contextual_action_similarity']
)

similarity_df = similarity_df.sort_values(
    by='contextual_score',
    ascending=False
).reset_index(drop=True)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'semantic_similarity',
            'action_similarity',
            'contextual_action_similarity',
            'contextual_score'
        ]
    ].head(25)
)

,event_1,event_2,semantic_similarity,action_similarity,contextual_action_similarity,contextual_score
0,7,21,0.825796,0.333333,0.916792,0.871294
1,21,32,0.788178,0.333333,0.860877,0.824528
2,7,32,0.775200,0.333333,0.834307,0.804753
3,1,13,0.762442,0.250000,0.845516,0.803979
4,0,13,0.784602,0.333333,0.820786,0.802694
5,16,39,0.770240,0.333333,0.832950,0.801595
6,7,20,0.773597,0.333333,0.775490,0.774543
7,13,26,0.698233,0.333333,0.830932,0.764583
8,0,26,0.757268,0.250000,0.763916,0.760592
9,1,26,0.738824,0.250000,0.780967,0.759895


In [18]:
for _, row in similarity_df.head(15).iterrows():

    i = int(row['event_1'])
    j = int(row['event_2'])

    e1 = events_df.iloc[i]
    e2 = events_df.iloc[j]

    print("=" * 110)
    print(f"EVENT PAIR: {i} and {j}")
    print(f"Articles: {e1['article_id']} and {e2['article_id']}")

    print(f"\nScores:")
    print(f"Semantic: {row['semantic_similarity']:.3f}")
    print(f"WordNet action: {row['action_similarity']:.3f}")
    print(f"Contextual action: {row['contextual_action_similarity']:.3f}")

    print("\nEVENT 1")
    print("Action:", e1['action_norm'])
    print("Subject:", e1['subject_norm'])
    print("Object:", e1['object_norm'])
    print("Location:", e1['location_norm'])
    print("Temporal:", e1['temporal_norm'])
    print("Sentence:", e1['original_sentence'])

    print("\nEVENT 2")
    print("Action:", e2['action_norm'])
    print("Subject:", e2['subject_norm'])
    print("Object:", e2['object_norm'])
    print("Location:", e2['location_norm'])
    print("Temporal:", e2['temporal_norm'])
    print("Sentence:", e2['original_sentence'])

EVENT PAIR: 7 and 21
Articles: article_01 and article_02

Scores:
Semantic: 0.826
WordNet action: 0.333
Contextual action: 0.917

EVENT 1
Action: open
Subject: ['the municipal corporation']
Object: ['two temporary shelters']
Location: ['central high school and the riverside community hall']
Temporal: []
Sentence: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.

EVENT 2
Action: set
Subject: ['temporary shelters']
Object: []
Location: ['central high school and riverside community hall']
Temporal: []
Sentence: The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.
EVENT PAIR: 21 and 32
Articles: article_02 and article_03

Scores:
Semantic: 0.788
WordNet action: 0.333
Contextual action: 0.861

EVENT 1
Action: set
Subject: ['temporary shelters']
Object: []
Location: ['central high school and riverside community hall']
Temporal: []
Sentence: The municipal

In [21]:
import spacy
nlp = spacy.load("en_core_web_sm")

sample_ids = [7, 21, 32, 16, 39]

for idx in sample_ids:

    row = events_df.iloc[idx]
    sentence = row['original_sentence']

    doc = nlp(sentence)

    print("=" * 100)
    print(f"Event ID: {idx}")
    print("Sentence:", sentence)
    print("Extracted action:", row['action_norm'])

    print("\nDependency parsing:")

    for token in doc:
        print(
            f"{token.text:15}",
            f"POS: {token.pos_:8}",
            f"DEP: {token.dep_:12}",
            f"HEAD: {token.head.text}"
        )

Event ID: 7
Sentence: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
Extracted action: open

Dependency parsing:
The             POS: DET      DEP: det          HEAD: corporation
municipal       POS: ADJ      DEP: amod         HEAD: corporation
corporation     POS: NOUN     DEP: nsubj        HEAD: opened
opened          POS: VERB     DEP: ROOT         HEAD: opened
two             POS: NUM      DEP: nummod       HEAD: shelters
temporary       POS: ADJ      DEP: amod         HEAD: shelters
shelters        POS: NOUN     DEP: dobj         HEAD: opened
at              POS: ADP      DEP: prep         HEAD: opened
Central         POS: PROPN    DEP: compound     HEAD: School
High            POS: PROPN    DEP: compound     HEAD: School
School          POS: PROPN    DEP: pobj         HEAD: at
and             POS: CCONJ    DEP: cc           HEAD: School
the             POS: DET      DEP: det          HEAD: Hall
Riverside       POS:

In [22]:
def extract_predicate(doc, main_verb):

    predicate_tokens = [main_verb]

    for child in main_verb.children:

        # Include phrasal verb particles: set up, carry out, etc.
        if child.dep_ == 'prt':
            predicate_tokens.append(child)

        # Include adjectival complements: remain closed, become operational
        elif child.dep_ == 'acomp':
            predicate_tokens.append(child)

        # Include verbal complements where appropriate
        elif child.dep_ in ['xcomp'] and child.pos_ == 'VERB':
            predicate_tokens.append(child)

    predicate_tokens = sorted(
        predicate_tokens,
        key=lambda token: token.i
    )

    return " ".join(token.lemma_.lower() for token in predicate_tokens)

In [24]:
def get_event_predicate(row):

    doc = nlp(row['original_sentence'])

    for token in doc:

        if (
            token.pos_ == 'VERB'
            and token.lemma_.lower() == row['action_norm']
        ):
            return extract_predicate(doc, token)

    return row['action_norm']


events_df['event_predicate'] = events_df.apply(
    get_event_predicate,
    axis=1
)

display(
    events_df[
        [
            'action_norm',
            'event_predicate',
            'original_sentence'
        ]
    ].head(40)
)

,action_norm,event_predicate,original_sentence
0,begin,begin,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
1,cause,cause,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
2,flood,flood,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
3,record,record,The city weather office recorded 118 millimetres of rain in twelve hours.
4,receive,receive,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.
5,enter,enter,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
6,divert,divert,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
7,open,open,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
8,move,move,Officials said that 46 residents had been moved to the shelters by 10 a.m.
9,warn,warn,The weather office warned that further rainfall was expected through Saturday.


In [25]:
def create_event_context(row):

    subject = " ".join(row['subject_norm'])
    predicate = row['event_predicate']
    obj = " ".join(row['object_norm'])
    location = " ".join(row['location_norm'])

    return f"{subject} {predicate} {obj} {location}".strip()


events_df['event_context'] = events_df.apply(
    create_event_context,
    axis=1
)

display(
    events_df[
        [
            'action_norm',
            'event_predicate',
            'event_context',
            'original_sentence'
        ]
    ].head(15)
)

,action_norm,event_predicate,event_context,original_sentence
0,begin,begin,continuous rainfall begin riverton the eastern part of the city,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
1,cause,cause,continuous rainfall cause riverton the eastern part of the city,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
2,flood,flood,several streets flood riverton the eastern part of the city,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
3,record,record,the city weather office record 118 millimetres,The city weather office recorded 118 millimetres of rain in twelve hours.
4,receive,receive,emergency services receive their first major flooding calls,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.
5,enter,enter,water enter ground floor homes the old bridge the riverside neighbourhood,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
6,divert,divert,traffic divert the old bridge the riverside neighbourhood,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
7,open,open,the municipal corporation open two temporary shelters central high school and the riverside community hall,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
8,move,move,46 residents move,Officials said that 46 residents had been moved to the shelters by 10 a.m.
9,warn,warn,the weather office warn,The weather office warned that further rainfall was expected through Saturday.


In [26]:
event_context_embeddings = model.encode(
    events_df['event_context'].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

event_context_matrix = cosine_similarity(event_context_embeddings)

similarity_df['improved_contextual_similarity'] = similarity_df.apply(
    lambda row: event_context_matrix[
        int(row['event_1']),
        int(row['event_2'])
    ],
    axis=1
)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'contextual_action_similarity',
            'improved_contextual_similarity',
            'semantic_similarity'
        ]
    ].head(25)
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

,event_1,event_2,contextual_action_similarity,improved_contextual_similarity,semantic_similarity
0,7,21,0.916792,0.923521,0.825796
1,21,32,0.860877,0.869488,0.788178
2,7,32,0.834307,0.834307,0.775200
3,1,13,0.845516,0.845516,0.762442
4,0,13,0.820786,0.820786,0.784602
5,16,39,0.832950,0.875445,0.770240
6,7,20,0.775490,0.775490,0.773597
7,13,26,0.830932,0.830932,0.698233
8,0,26,0.763916,0.763916,0.757268
9,1,26,0.780967,0.780967,0.738824


In [27]:
sample_ids = [3, 30, 7, 21, 32, 16, 39]

for idx in sample_ids:

    row = events_df.iloc[idx]

    print("=" * 100)
    print("Event ID:", idx)
    print("Article:", row['article_id'])
    print("Original action:", row['action_norm'])
    print("Improved predicate:", row['event_predicate'])
    print("Subject:", row['subject_norm'])
    print("Object:", row['object_norm'])
    print("Location:", row['location_norm'])
    print("Sentence:", row['original_sentence'])

Event ID: 3
Article: article_01
Original action: record
Improved predicate: record
Subject: ['the city weather office']
Object: ['118 millimetres']
Location: []
Sentence: The city weather office recorded 118 millimetres of rain in twelve hours.
Event ID: 30
Article: article_03
Original action: receive
Improved predicate: receive
Subject: ['the city']
Object: ['118 millimetres']
Location: ['the twelve hours leading up to friday morning']
Sentence: The weather office reported that the city had received 118 millimetres of rain in the twelve hours leading up to Friday morning.
Event ID: 7
Article: article_01
Original action: open
Improved predicate: open
Subject: ['the municipal corporation']
Object: ['two temporary shelters']
Location: ['central high school and the riverside community hall']
Sentence: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
Event ID: 21
Article: article_02
Original action: set
Improved predicate: set

In [28]:
sample_ids = [3, 30, 7, 21, 32, 16, 39]

for idx in sample_ids:

    row = events_df.iloc[idx]

    print("=" * 100)
    print("Event ID:", idx)
    print("Sentence:", row['original_sentence'])

    for col in [
        'subject_norm',
        'object_norm',
        'agent',
        'passive',
        'event_predicate',
        'location_norm',
        'temporal_norm'
    ]:
        if col in events_df.columns:
            print(f"{col}: {row[col]}")

Event ID: 3
Sentence: The city weather office recorded 118 millimetres of rain in twelve hours.
subject_norm: ['the city weather office']
object_norm: ['118 millimetres']
agent: []
passive: False
event_predicate: record
location_norm: []
temporal_norm: ['twelve hours']
Event ID: 30
Sentence: The weather office reported that the city had received 118 millimetres of rain in the twelve hours leading up to Friday morning.
subject_norm: ['the city']
object_norm: ['118 millimetres']
agent: []
passive: False
event_predicate: receive
location_norm: ['the twelve hours leading up to friday morning']
temporal_norm: ['friday', 'morning', 'twelve hours']
Event ID: 7
Sentence: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
subject_norm: ['the municipal corporation']
object_norm: ['two temporary shelters']
agent: []
passive: False
event_predicate: open
location_norm: ['central high school and the riverside community hall']
temporal_nor

In [29]:
def extract_semantic_roles(sentence, predicate):

    doc = nlp(sentence)

    agent = []
    patient = []

    for token in doc:

        # Find the main event predicate
        if (
            token.pos_ == 'VERB'
            and token.lemma_.lower() in predicate.split()
        ):

            # Active voice
            if token.dep_ != 'ROOT' or True:

                for child in token.children:

                    # Active subject
                    if child.dep_ == 'nsubj':
                        agent.append(child.subtree)

                    # Active object
                    elif child.dep_ in ['dobj', 'obj']:
                        patient.append(child.subtree)

                    # Passive subject
                    elif child.dep_ in ['nsubjpass', 'nsubj:pass']:
                        patient.append(child.subtree)

                    # Explicit passive agent
                    elif child.dep_ == 'agent':
                        for agent_child in child.children:
                            if agent_child.dep_ == 'pobj':
                                agent.append(agent_child.subtree)

    def convert_to_text(items):

        return [
            " ".join(token.text for token in subtree).lower()
            for subtree in items
        ]

    return convert_to_text(agent), convert_to_text(patient)

In [30]:
roles = events_df.apply(
    lambda row: extract_semantic_roles(
        row['original_sentence'],
        row['event_predicate']
    ),
    axis=1
)

events_df['agent_norm'] = roles.apply(lambda x: x[0])
events_df['patient_norm'] = roles.apply(lambda x: x[1])

display(
    events_df[
        [
            'event_predicate',
            'agent_norm',
            'patient_norm',
            'original_sentence'
        ]
    ].head(20)
)

,event_predicate,agent_norm,patient_norm,original_sentence
0,begin,[continuous rainfall],[],"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
1,cause,[],[],"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
2,flood,[several streets in the eastern part of the city],[],"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn..."
3,record,[the city weather office],[118 millimetres of rain],The city weather office recorded 118 millimetres of rain in twelve hours.
4,receive,[emergency services],[their first major flooding calls],Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.
5,enter,[water],[ground - floor homes],"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
6,divert,[],[traffic],"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
7,open,[the municipal corporation],[two temporary shelters],The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.
8,move,[],[46 residents],Officials said that 46 residents had been moved to the shelters by 10 a.m.
9,warn,[the weather office],[],The weather office warned that further rainfall was expected through Saturday.


In [31]:
def entity_similarity(a, b):
    if not a or not b:
        return 0.0

    a = set(a)
    b = set(b)

    return len(a & b) / len(a | b)


similarity_df['agent_similarity'] = similarity_df.apply(
    lambda row: entity_similarity(
        events_df.iloc[int(row['event_1'])]['agent_norm'],
        events_df.iloc[int(row['event_2'])]['agent_norm']
    ),
    axis=1
)

similarity_df['patient_similarity'] = similarity_df.apply(
    lambda row: entity_similarity(
        events_df.iloc[int(row['event_1'])]['patient_norm'],
        events_df.iloc[int(row['event_2'])]['patient_norm']
    ),
    axis=1
)

In [32]:
display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'semantic_similarity',
            'contextual_action_similarity',
            'improved_contextual_similarity',
            'agent_similarity',
            'patient_similarity'
        ]
    ].sort_values(
        'improved_contextual_similarity',
        ascending=False
    ).head(25)
)

,event_1,event_2,semantic_similarity,contextual_action_similarity,improved_contextual_similarity,agent_similarity,patient_similarity
0,7,21,0.825796,0.916792,0.923521,0.0,0.0
5,16,39,0.770240,0.832950,0.875445,0.0,0.0
1,21,32,0.788178,0.860877,0.869488,0.0,0.0
3,1,13,0.762442,0.845516,0.845516,0.0,0.0
2,7,32,0.775200,0.834307,0.834307,0.0,0.0
7,13,26,0.698233,0.830932,0.830932,0.0,0.0
4,0,13,0.784602,0.820786,0.820786,0.0,0.0
9,1,26,0.738824,0.780967,0.780967,0.0,0.0
6,7,20,0.773597,0.775490,0.775490,1.0,0.0
8,0,26,0.757268,0.763916,0.763916,0.0,0.0


In [33]:
similarity_df.columns

Index(['event_1', 'event_2', 'article_1', 'article_2', 'sentence_1',
       'sentence_2', 'semantic_similarity', 'subject_similarity',
       'object_similarity', 'location_similarity', 'temporal_similarity',
       'action_similarity', 'combined_similarity',
       'contextual_action_similarity', 'contextual_score',
       'improved_contextual_similarity', 'agent_similarity',
       'patient_similarity'],
      dtype='str')

In [44]:
similarity_df['combined_similarity'] = (
    0.25 * similarity_df['semantic_similarity'] +
    0.25 * similarity_df['improved_contextual_similarity'] +
    0.15 * similarity_df['action_similarity'] +
    0.15 * similarity_df['patient_similarity'] +
    0.15 * similarity_df['location_similarity'] +
    0.05 * similarity_df['temporal_similarity']
)

similarity_df = similarity_df.sort_values(
    'combined_similarity',
    ascending=False
).reset_index(drop=True)

display(
    similarity_df[
        [
            'event_1',
            'event_2',
            'combined_similarity',
        ]
    ].head(25)
)


,event_1,event_2,combined_similarity
0,13,27,0.661432
1,7,21,0.618579
2,21,32,0.614416
3,13,26,0.601041
4,3,30,0.598667
5,7,32,0.583627
6,7,20,0.568522
7,6,17,0.564297
8,1,29,0.543910
9,13,29,0.539255


In [45]:
SIMILARITY_THRESHOLD = 0.5

matched_pairs = similarity_df[
    (similarity_df['combined_similarity'] >= SIMILARITY_THRESHOLD) &
    (similarity_df['article_1'] != similarity_df['article_2'])
].copy()

matched_pairs = matched_pairs.sort_values(
    'combined_similarity',
    ascending=False
).reset_index(drop=True)

print("Total event pairs:", len(similarity_df))
print("Matched event pairs:", len(matched_pairs))

display(matched_pairs.head(20))

Total event pairs: 585
Matched event pairs: 12


,event_1,event_2,article_1,article_2,sentence_1,sentence_2,semantic_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity,action_similarity,combined_similarity,contextual_action_similarity,contextual_score,improved_contextual_similarity,agent_similarity,patient_similarity
0,13,27,article_02,article_03,"Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.","Rainfall eased in Riverton on Saturday, August 15, but several neighbourhoods continued to deal with flooding caused by the previous day’s downpour.",0.810595,0.0,0.0,1.000000,0.375,1.000000,0.661432,0.637284,0.723939,0.560134,0.0,0.0
1,7,21,article_01,article_02,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,0.825796,0.0,0.0,0.875000,0.000,0.333333,0.618579,0.916792,0.871294,0.923521,0.0,0.0
2,21,32,article_02,article_03,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,The municipal corporation said that 60 residents had used the temporary shelters at Central High School and Riverside Community Hall.,0.788178,0.0,0.0,1.000000,0.000,0.333333,0.614416,0.860877,0.824528,0.869488,0.0,0.0
3,13,26,article_02,article_03,"Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.","Rainfall eased in Riverton on Saturday, August 15, but several neighbourhoods continued to deal with flooding caused by the previous day’s downpour.",0.698233,0.5,0.0,1.000000,0.375,0.333333,0.601041,0.830932,0.764583,0.830932,0.0,0.0
4,3,30,article_01,article_03,The city weather office recorded 118 millimetres of rain in twelve hours.,The weather office reported that the city had received 118 millimetres of rain in the twelve hours leading up to Friday morning.,0.844065,0.5,1.0,0.000000,0.500,0.333333,0.598667,0.650604,0.747334,0.650604,0.0,1.0
5,7,32,article_01,article_03,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,The municipal corporation said that 60 residents had used the temporary shelters at Central High School and Riverside Community Hall.,0.775200,0.0,0.5,0.875000,0.000,0.333333,0.583627,0.834307,0.804753,0.834307,0.0,0.0
6,7,20,article_01,article_02,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,The municipal corporation confirmed that temporary shelters had been set up at Central High School and Riverside Community Hall.,0.773597,1.0,0.0,0.875000,0.000,0.333333,0.568522,0.775490,0.774543,0.775490,1.0,0.0
7,6,17,article_01,article_02,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.","Police redirected traffic toward the North Bypass, while emergency teams helped residents leave low-lying homes in Riverside.",0.666000,0.0,0.0,0.285714,0.000,0.333333,0.564297,0.619760,0.642880,0.619760,0.0,1.0
8,1,29,article_01,article_03,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morn...","Rainfall eased in Riverton on Saturday, August 15, but several neighbourhoods continued to deal with flooding caused by the previous day’s downpour.",0.844869,0.0,0.0,0.166667,0.100,1.000000,0.543910,0.610771,0.727820,0.610771,0.0,0.0
9,13,29,article_02,article_03,"Heavy rainfall continued in Riverton on Friday, August 14, leading authorities to close all municipal schools for the day.","Rainfall eased in Riverton on Saturday, August 15, but several neighbourhoods continued to deal with flooding caused by the previous day’s downpour.",0.713094,0.0,0.0,1.000000,0.375,0.250000,0.539255,0.618925,0.6660

In [46]:
for idx, row in events_df.iterrows():

    if row['temporal_norm']:

        print("=" * 80)
        print("Event:", idx)
        print("Article:", row['article_id'])
        print("Sentence:", row['original_sentence'])
        print("Temporal:", row['temporal_norm'])

Event: 0
Article: article_01
Sentence: Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.
Temporal: ['august 13', 'early friday morning']
Event: 1
Article: article_01
Sentence: Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.
Temporal: ['august 13', 'early friday morning']
Event: 2
Article: article_01
Sentence: Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.
Temporal: ['august 13', 'early friday morning']
Event: 3
Article: article_01
Sentence: The city weather office recorded 118 millimetres of rain in twelve hours.
Temporal: ['twelve hours']
Event: 4
Article: article_01
Sentence: Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.
Tempo